# Dynamic hierarchical points model: evaluation and live-state monitoring

The model separates immediately observable role and context from a slowly changing player state:

\[
a_{i,t}=a_{i,t-1}+\epsilon_{i,t}, \qquad
y_{i,t}=x_{i,t}^{\mathsf T}\beta+\alpha_{\mathrm{position}}+\alpha_{\mathrm{team}}+a_{i,t}+\eta_{i,t}.
\]

Minutes, shot volume, rest, opponent defense, home court, position, and team determine the shared pregame component. A Kalman update moves the latent state only after a completed game's points are observed. The configured 0.35-point game drift and 2.0-point offseason drift are explicit assumptions, not values selected on an evaluation season.

**Status and leakage contract.** Each walk-forward fold fits on the two complete seasons immediately preceding its evaluation season. Lagged role features, opponent defense, and player state use only information available before each game; the current game's points enter only the postgame state update. Fold audits and causal assertions below make that contract visible.

**Two distinct outputs.** The multi-season walk-forward trace is an out-of-sample evaluation. The final section rebuilds the latest-season state causally from the latest two complete prior seasons and shows each player's most recently updated state. It is monitoring of observed live state, **not a future-game forecast**: no future schedule, opponent, or role row is supplied.

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("run this notebook from inside the nba_quant repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.dynamic_points import (
    DynamicPointsConfig,
    build_causal_frame,
    walk_forward_trace,
)
from src.models.dynamic_points_diagnostics import (
    core_calibration,
    gain_summary,
    group_calibration,
    offseason_summary,
    posterior_draw_summary,
    shrinkage_summary,
    variance_decomposition,
)

sns.set_theme(style="whitegrid")
CONFIG = DynamicPointsConfig(
    state_drift_sd=0.35,
    offseason_drift_sd=2.0,
    train_window_seasons=2,
    min_group_n=30,
)
SEASON_PATHS = sorted(
    ROOT.glob("data/silver/nba/*/regular_season/player_gamelogs.parquet")
)
if len(SEASON_PATHS) < CONFIG.train_window_seasons + 1:
    raise ValueError("need at least three complete season parquet files")

print(f"Repository root: {ROOT}")
print(f"Discovered {len(SEASON_PATHS)} season files")

In [ ]:
season_frames = []
raw_rows = 0
for season_path in SEASON_PATHS:
    season = pd.read_parquet(season_path)
    raw_rows += len(season)
    season = season.loc[season["season_type"].eq("Regular Season")].copy()
    season_frames.append(season)

games = pd.concat(season_frames, ignore_index=True)
missing_position_rows = int(games["pos"].isna().sum())
games["pos"] = games["pos"].fillna("UNK").astype(str)
print(
    f"Loaded {raw_rows:,} rows; retained {len(games):,} regular-season rows "
    f"from {len(SEASON_PATHS)} files."
)
print(
    f"Normalized {missing_position_rows:,} missing position values to the "
    "explicit 'UNK' category; no rows were dropped."
)

causal, causal_audit = build_causal_frame(games)
print("Causal-frame audit:")
display(pd.DataFrame([causal_audit]))

available_seasons = sorted(
    causal["season_year"].astype(str).unique(),
    key=lambda value: int(value.split("-")[0]),
)
evaluation_seasons = tuple(available_seasons[CONFIG.train_window_seasons :])
trace, fold_audits = walk_forward_trace(
    causal,
    evaluation_seasons=evaluation_seasons,
    config=CONFIG,
)
fold_audit_table = pd.DataFrame(fold_audits)
print("Walk-forward fold audits (every evaluation fold):")
display(fold_audit_table)

if trace.empty:
    raise AssertionError("walk-forward evaluation trace must not be empty")
numeric_trace = trace.select_dtypes(include="number")
if numeric_trace.empty or not np.isfinite(numeric_trace.to_numpy(dtype=float)).all():
    raise AssertionError("numeric diagnostic columns must be finite")
trace_key = ["season_year", "player_id", "game_id"]
if trace.duplicated(trace_key).any():
    raise AssertionError("trace must contain one row per player/game/fold")
if set(trace["season_year"].unique()) != set(evaluation_seasons):
    raise AssertionError("every requested evaluation season must appear in the trace")
print(
    f"Validated {len(trace):,} causal predictions across "
    f"{len(evaluation_seasons)} evaluation seasons: {', '.join(evaluation_seasons)}"
)

In [ ]:
core = core_calibration(trace)
core_table = pd.DataFrame(
    [{key: value for key, value in core.items() if key != "pit_histogram"}]
)
print("Core calibration:")
display(core_table)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(np.arange(10), core["pit_histogram"], width=0.9)
ax.axhline(core["n"] / 10, color="black", linestyle="--", label="Uniform reference")
ax.set(
    title=(
        f"PIT histogram — n={core['n']:,}; "
        f"seasons={', '.join(core['seasons'])}"
    ),
    xlabel="PIT decile (0=lowest)",
    ylabel="Predictions",
)
ax.legend()
plt.show()

variance_report = variance_decomposition(trace)
print("Variance decomposition and coverage by games-in-model tier:")
display(variance_report)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
coverage_columns = ["coverage_50", "coverage_80", "coverage_90", "coverage_95"]
variance_report.set_index("games_tier")[coverage_columns].plot(kind="bar", ax=axes[0])
for level in (0.50, 0.80, 0.90, 0.95):
    axes[0].axhline(level, color="gray", linewidth=0.7, linestyle="--")
axes[0].set(
    title=f"Empirical coverage by experience tier — n={len(trace):,}; {', '.join(evaluation_seasons)}",
    ylabel="Coverage",
    xlabel="Games in model",
)
axes[0].tick_params(axis="x", rotation=0)
axes[1].plot(variance_report["games_tier"].astype(str), variance_report["mean_z"], marker="o", label="Mean z")
axes[1].plot(variance_report["games_tier"].astype(str), variance_report["sd_z"], marker="s", label="SD z")
axes[1].axhline(0, color="black", linestyle="--", label="Mean z reference")
axes[1].axhline(1, color="gray", linestyle=":", label="SD z reference")
axes[1].set(title=f"Standardized residuals — n={len(trace):,}; {', '.join(evaluation_seasons)}", xlabel="Games in model")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
tier_shrinkage, state_decile_shrinkage = shrinkage_summary(trace)
print("Shrinkage by games-in-model tier:")
display(tier_shrinkage)
print("Shrinkage by pregame-state decile:")
display(state_decile_shrinkage)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(tier_shrinkage["games_tier"].astype(str), tier_shrinkage["mean_z"], marker="o")
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set(
    title=f"Mean standardized innovation by experience tier — n={len(trace):,}; {', '.join(evaluation_seasons)}",
    xlabel="Games in model",
    ylabel="Mean z",
)
axes[1].errorbar(
    state_decile_shrinkage["mean_state"],
    state_decile_shrinkage["mean_innovation"],
    yerr=2 * state_decile_shrinkage["innovation_se"],
    fmt="o",
    capsize=3,
)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].axvline(0, color="gray", linestyle=":")
axes[1].set(
    title=f"State-decile shrinkage — n={len(trace):,}; {', '.join(evaluation_seasons)}",
    xlabel="Mean state before game",
    ylabel="Mean innovation (±2 SE)",
)
plt.tight_layout()
plt.show()

In [ ]:
offseason_windows, innovation_state_relationship = offseason_summary(trace)
print("Offseason calibration for all requested early windows and their complements:")
display(offseason_windows)
print("Early-season innovation/state relationship (returners, games 1-20):")
display(pd.DataFrame([innovation_state_relationship]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for period, subset in offseason_windows.groupby("period", sort=False):
    axes[0].plot(subset["window"], subset["mean_z"], marker="o", label=period)
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set(
    title=f"Returner calibration by early-season window — n={int(trace['returning_player'].sum()):,}; {', '.join(evaluation_seasons)}",
    xlabel="Window (games)",
    ylabel="Mean z",
)
axes[0].legend()
early_returners = trace.loc[trace["returning_player"] & trace["season_game_number"].le(20)]
sns.regplot(data=early_returners, x="state_before", y="innovation", scatter_kws={"alpha": 0.15, "s": 10}, ax=axes[1])
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(
    title=f"Early innovation vs. state — n={len(early_returners):,}; {', '.join(evaluation_seasons)}",
    xlabel="State before game",
    ylabel="Innovation",
)
plt.tight_layout()
plt.show()

In [ ]:
group_reports = {}
for group_column in ("opp_team_id", "team_id", "position"):
    report = group_calibration(trace, group_column, min_n=CONFIG.min_group_n)
    report = report.sort_values(
        ["actionable_flag", "low_sample", "n"],
        ascending=[False, True, False],
        kind="mergesort",
    ).reset_index(drop=True)
    group_reports[group_column] = report
    print(f"Calibration by {group_column} (actionable flags first; low-sample rows retained):")
    display(report)

slice_reports = {}
for flag_column, label in (("new_team_flag", "New-team slice"), ("rookie_flag", "Rookie slice")):
    slice_report = group_calibration(trace, flag_column, min_n=CONFIG.min_group_n)
    slice_report = slice_report.sort_values("actionable_flag", ascending=False, kind="mergesort")
    slice_reports[flag_column] = slice_report
    print(f"{label} (False and True shown separately):")
    display(slice_report)

In [ ]:
gain_by_tier, gain_by_games, gain_by_boundary = gain_summary(trace)
print("Kalman gain by broad experience tier:")
display(gain_by_tier)
print("Kalman gain by fine games-in-model bins:")
display(gain_by_games)
print("Kalman gain by season-boundary status:")
display(gain_by_boundary)

fig, ax = plt.subplots(figsize=(10, 5))
x = (gain_by_games["games_min"] + gain_by_games["games_max"]) / 2
ax.errorbar(x, gain_by_games["mean_gain"], yerr=2 * gain_by_games["gain_se"], marker="o", capsize=3, label="Fine-bin mean (±2 SE)")
for row in gain_by_boundary.itertuples(index=False):
    ax.axhline(row.mean_gain, linestyle="--" if row.season_boundary else ":", label=f"Season boundary={row.season_boundary} mean")
ax.set(
    title=f"Kalman gain vs. games in model — n={len(trace):,}; {', '.join(evaluation_seasons)}",
    xlabel="Games in model (bin midpoint)",
    ylabel="Mean Kalman gain",
)
ax.legend()
plt.show()

unexplained_jumps = gain_by_games.loc[gain_by_games["unexplained_jump_flag"]]
print("Unexplained positive non-boundary gain jumps:")
if unexplained_jumps.empty:
    print("None")
else:
    display(unexplained_jumps)

draw_report = posterior_draw_summary(trace, draws=None)
print("Posterior-draw diagnostics:", draw_report)

In [ ]:
interpretation_lines = ["Diagnostic interpretation and follow-up:"]

for group_column, report in group_reports.items():
    flagged = report.loc[report["actionable_flag"]]
    labels = flagged[group_column].astype(str).tolist()
    preview = ", ".join(labels[:5])
    remainder = len(labels) - min(len(labels), 5)
    preview_text = (
        f"; first flags: {preview}" + (f" (+{remainder} more)" if remainder else "")
        if preview
        else ""
    )
    interpretation_lines.append(
        f"- {group_column}: {len(flagged)} actionable calibration flag(s){preview_text}."
    )

for flag_column, label in (("new_team_flag", "New-team"), ("rookie_flag", "Rookie")):
    report = slice_reports[flag_column]
    flagged_slice = report.loc[report[flag_column] & report["actionable_flag"]]
    true_slice = report.loc[report[flag_column]]
    if true_slice.empty:
        interpretation_lines.append(f"- {label}: no evaluated rows were available.")
    else:
        row = true_slice.iloc[0]
        flag_status = "actionable calibration flag" if not flagged_slice.empty else "no actionable calibration flag"
        interpretation_lines.append(
            f"- {label}: {flag_status}; n={int(row['n']):,}, "
            f"mean z={row['mean_z']:.3f}, SD z={row['sd_z']:.3f}."
        )

slope = float(innovation_state_relationship["slope"])
p_value = float(innovation_state_relationship["p_value"])
if np.isfinite(slope) and np.isfinite(p_value):
    relationship_label = "mean-reversion signal" if slope < 0 else "same-direction signal"
    significance_label = "statistically resolved" if p_value < 0.05 else "not statistically resolved"
    interpretation_lines.append(
        f"- Early-season state/innovation relationship: {relationship_label}; "
        f"slope={slope:.3f}, p={p_value:.3g} ({significance_label})."
    )
else:
    interpretation_lines.append("- Early-season state/innovation relationship: insufficient variation for inference.")

if unexplained_jumps.empty:
    interpretation_lines.append("- Non-boundary gain jumps: none exceeded the diagnostic threshold.")
else:
    jump_bins = ", ".join(
        f"{int(row.games_min)}-{int(row.games_max)} games"
        for row in unexplained_jumps.itertuples(index=False)
    )
    interpretation_lines.append(
        f"- Non-boundary gain jumps: {len(unexplained_jumps)} flagged bin(s) ({jump_bins}); investigate before operational use."
    )

interpretation_lines.append(
    "These diagnostics identify follow-up checks only; they do not constitute deployment approval."
)
print("\n".join(interpretation_lines))

## Latest-season live-state monitoring

This section uses the latest two complete prior seasons as the training window and processes the latest available season sequentially with the same causal API. The table is a snapshot of each player's state after their most recently observed game. It is **monitoring, not a future-game forecast**: there is no future schedule, opponent, or role row from which to construct a pregame prediction.

In [ ]:
latest_season = available_seasons[-1]
prior_seasons = available_seasons[:-1]
if len(prior_seasons) < CONFIG.train_window_seasons:
    raise ValueError("latest-season monitoring needs two complete prior seasons")
live_training_seasons = tuple(prior_seasons[-CONFIG.train_window_seasons :])
live_seasons = (*live_training_seasons, latest_season)
live_frame = causal.loc[causal["season_year"].astype(str).isin(live_seasons)].copy()
live_trace, live_fold_audits = walk_forward_trace(
    live_frame,
    evaluation_seasons=(latest_season,),
    config=CONFIG,
)
print(
    f"Live-state window: train on {', '.join(live_training_seasons)}; "
    f"causally monitor {latest_season}."
)
print("Latest-season fold audit:")
display(pd.DataFrame(live_fold_audits))

if live_trace.empty:
    raise AssertionError("latest-season live-state trace must not be empty")
latest_rows = (
    live_trace.sort_values(["player_id", "game_date", "game_id"], kind="mergesort")
    .groupby("player_id", sort=False, as_index=False)
    .tail(1)
    .copy()
)
latest_rows["state_sd_after"] = np.sqrt(latest_rows["state_var_after"])
latest_rows["games_in_model"] = latest_rows["games_in_model_before"] + 1
monitoring_columns = [
    "player_name",
    "player_id",
    "team_id",
    "game_date",
    "state_after",
    "state_sd_after",
    "games_in_model",
    "predictive_mean",
    "points",
    "innovation",
    "z",
    "pit",
    "kalman_gain",
]
latest_state_monitoring = latest_rows.loc[:, monitoring_columns].sort_values(
    ["state_after", "player_name"], ascending=[False, True], kind="mergesort"
)
if not np.isfinite(latest_state_monitoring.select_dtypes(include="number").to_numpy(dtype=float)).all():
    raise AssertionError("latest-season monitoring diagnostics must be finite")
print(
    f"Latest-season state monitoring (n={len(latest_state_monitoring):,} players). "
    "Observed-state snapshot; not a future-game forecast."
)
display(latest_state_monitoring.reset_index(drop=True))